In [1]:
import urllib.error
import urllib.request
import pprint
from langchain.tools import tool

from langchain.chat_models import init_chat_model

import langchain_groq
import os

from dotenv import load_dotenv

load_dotenv()

True

In [2]:

model_groq_lamma70b = init_chat_model("llama-3.3-70b-versatile",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=10000, temperature=0.0)

# model = init_chat_model("openai/gpt-4o-mini",
#                         api_key=os.environ["OPENROUTER_API_KEY"],
#                         model_provider="openrouter",
#                         base_url="https://openrouter.ai/api/v1",
#                         max_tokens=1000, temperature=0.0)

# model = init_chat_model("nvidia/nemotron-3-ultra-550b-a55b:free",
#                         api_key=os.environ["OPENROUTER_API_KEY"],
#                         model_provider="openrouter",
#                         base_url="https://openrouter.ai/api/v1",
#                         max_tokens=1000, temperature=0.0)


# model = init_chat_model("openrouter/free",
#                         api_key=os.environ["OPENROUTER_API_KEY"],
#                         model_provider="openrouter",
#                         base_url="https://openrouter.ai/api/v1",
#                         max_tokens=1000, temperature=0.0)


model_or = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)

model_or_paid_gpt56_luna_pro = init_chat_model("openai/gpt-5.6-luna-pro",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)

In [3]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable


@tool()
def public_search(query: str) -> str:
    """Perform a public search and return the results."""
    # In production, you would call a search API here
    return f"Search results for '{query}': [Result 1, Result 2, Result 3]"

@tool()
def private_search(query: str) -> str:
    """Perform a private search and return the results."""
    # In production, you would call a private search API here
    return f"Private search results for '{query}': [Private Result 1, Private Result 2]"

@tool()
def advanced_search(query: str) -> str:
    """Perform an advanced search and return the results."""
    # In production, you would call an advanced search API here
    return f"Advanced search results for '{query}': [Advanced Result 1, Advanced Result 2]"


In [5]:

@wrap_model_call
def state_based_tools(request : ModelRequest,
                      handler : Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Filter tools based on conversation State."""
    # Read from State: check if user has authenticated
    state = request.state
    is_authenticated = state.get("authenticated", False)
    message_count = len(state["messages"])

    # Only enable sensitive tools after authentication
    if not is_authenticated:
        tools = [t for t in request.tools if t.name.startswith("public_")]
        request = request.override(tools=tools)
    elif message_count < 5:
        # Limit tools early in conversation
        tools = [t for t in request.tools if t.name != "advanced_search"]
        request = request.override(tools=tools)

    return handler(request)


In [6]:
agent = create_agent(
    model=model_or_paid_gpt56_luna_pro,
    tools=[public_search, private_search, advanced_search],
    middleware=[state_based_tools]
)

In [7]:
from rich import print as rprint

result = agent.invoke({
    "messages": [{"role": "user", "content": "do a public search on Sports"}]
})

rprint(result)

{
    'messages': [
        HumanMessage(
            content='do a public search on Sports',
            additional_kwargs={},
            response_metadata={},
            id='44985edf-a91a-4118-a2f8-6fc02357ac11'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'data': 
'gAAAAABqgIhPCPi2YXO-i2Ufu7GcTlThqQQKcxu7DeuFvHR-dWl0erPzCmlXZMolEgedxLTSn9tcMi97F49YIqCNI2XA9pvy_iwe6KwysjpKYNOJKX
wO-WJMvJCSAko2K4IyAi7oedUXd5GHzRtyOk1HZKMh_jAvAotLvb9fh5NDtYaMjCs3vw8taQ-cwXPdRjaWW5XiEvVm_EniKG0iKDj_AVaciulfE7rcQ
MSnGfN8StmTSdUKtKmRUltbOgIhPAvzeANyhYwHKsC79_2JGmuLP0K3sRnSr-__fgC57KQLDpCRjzsZRjxnwmLDUGYM57nymXTHX1uifZRIfeN2GF88
EiRTHjsPVBHYkJM0PqTLUOselOuOuxncCFVcODObDsmcC2VCx_BB0K6-q75RfRHzJ0IgEuxFnHGbc5kVvE2-774LpHajxlWcOABWXxyHk3FEzlfxfXt
xbevuUCeo68sBhoLdQJjX25cNHzXgG-RIeZ9uXbAUDi5dWpznDFGGQzVx99Bpf9f88zPMLegyn8300WHiZeLxSiHG3SQtS7cCztG7gcTAgb9ViCkOAQ
2FAAcmLnaF0_TgjqTQzH3b5phHtcsJ4z6iQyY3nia1UIJelTzE8XMwePFnbpekbB0P76nDoceluqm9M2MqEr_4JaZywM4IkTgOgYp-DWDBv5tfSKRWU
yxUKdqp-Rx6OCURK90uS3iH2ZZGcRa1DGI8G4sus8oH4Ehe1OucIU2kFl50mXKE6CzM8DQ6auBwm6LZjvLQBX3EsVs6Dngs8Sf92Gm4XfiS4-3XbIzP
mup0uDLKrBIb6gBAyYdgnrPPdCTGM2QLP0eefIYu9GEBImWRJteFSmXwqo0mVWdP9KIayWWnkTizQ9zIfRm5yC5VdtZJNyp7XeuD02SaYSUzezGnjhO
-B5GAbPQzMQnsQ4UTgwAbbq9cFK1tSM7JDeH-FuYYgtskVh-GnWsFRj2h6CFCUtbtpAI4-xjkZVoDY4V4j6oIIdKJ4HUElb8EnHMktXItq9VifFwbnb
5pqjZs4EccD3m8CLXuFlOs0AFlMoyOY8b1YEXex0yLAEVPnD7qkV-YdlBL9a2aRz9-qjBBoBkt1gImRIsxbu3o042hAvJT6pXMoI7KbEemgPoCMFwM-
SLFieHDmWgwl95J0mz8Hm80uqTcSDouBF8l5Hhzbv323WcIqCz0Uj89gK5dwOkp2tIJXtCei-AHroITOsNte-K-1KCIRUdYcF0ERKZBUdM0hy6CwJJF
pExE0gRwEt-UvyAxAmoZZld7BUHULjHoZq_LCSwEKHuhHhBKeZem9Su2g04uLDfz3Cdv7vRPdRMIQ0Ze3nYqz21hD4CpB7DFqbSIZntwQp04CPgW06_
LW6pIZ4Qe24VWE_cZBC-ikZqdGj1F_fRc8qPjfpalqYjYgORzMEpdGfZaHXbFwQ==.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dwdC01LjYtbHVuY
S1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_0202081d95b97483016a80884f24188190b9a75bffc1a31ce5',
                        'index': 0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786808396-kJziFqYssgBn5c6jGHgY',
                'created': 1786808396,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0002723,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0001038,
                    'upstream_inference_prompt_cost': 0.0001685,
                    'upstream_inference_cost': 0.0002723
                }
            },
            id='lc_run--01a00614-61e3-7200-874a-48445a9c7958-0',
            tool_calls=[
                {
                    'name': 'public_search',
                    'args': {'query': 'Sports'},
                    'id': 'call_Q2Sk4FpDupivM9BqEEQcrTBU',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1685,
                'output_tokens': 173,
                'total_tokens': 1858,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 96}
            }
        ),
        ToolMessage(
            content="Search results for 'Sports': [Result 1, Result 2, Result 3]",
            name='public_search',
            id='5f7796df-b0b3-4783-a376-997aebc0e13a',
            tool_call_id='call_Q2Sk4FpDupivM9BqEEQcrTBU'
        ),
        AIMessage(
            content='Public search results for **“Sports”**:\n\n- Result 1\n- Result 2\n- Result 3',
            additional_kwargs={},
            response_metadat

In [8]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "do a search on politics"}]
})

rprint(result)

{
    'messages': [
        HumanMessage(
            content='do a search on politics',
            additional_kwargs={},
            response_metadata={},
            id='a803f39b-4977-4ba4-bf8c-05569fc20712'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_content': "**Searching for news clarification**\n\nI think we need to clarify the type 
of news we're looking for. A broad search might be helpful, especially if we're focusing on politics. I can 
initiate a public search query to gather some recent political news that might interest the user. Let’s see what 
the latest headlines are and get a clearer picture for them!",
                'reasoning_details': [
                    {
                        'summary': "**Searching for news clarification**\n\nI think we need to clarify the type of 
news we're looking for. A broad search might be helpful, especially if we're focusing on politics. I can initiate a
public search query to gather some recent political news that might interest the user. Let’s see what the latest 
headlines are and get a clearer picture for them!",
                        'type': 'reasoning.summary',
                        'format': 'openai-responses-v1',
                        'index': 0
                    },
                    {
                        'data': 
'gAAAAABqgIiU5pZY2TnCPGg1hjubIfDFeZKwznnoFAo6Ivfc2vGkFFkARGkNL0Xt9PEDMSO0GT8KuPH5vDAEqb3a_YPAS9jbDD31oykui7P1r1-51C
sXGmhDsHVB7oZCPCT_s-q2SrnNX_xpI0Dou60FhUCfwDfEdaolBaXYcw1X6AHw_U9bC9Df4ws9pofZCifqjFD0StvFpmGapSeoH2sBk5VfoRakpdYFK
o6jJUWqV99aZicwEvpyAHWt4WhGjfHkq1l_d5aysjQtSGiTdlbGlfDmMRKjsdk8reuvlAbrf4mG8_wogKG-11UJqixZob4UafI2ddT0Ud_Tn24X-MPv
g__5gfqroYsm-s6wU1VfI6j7d3ga8UCphq_9l5yDwMX7HwFZD_AxKJ_6nIQZYlJ0gXhyfmvq25-uZrAVu0c13rzrF-FATn5mFyiy5QcujrL1D-rJF-M
RpPe1fvQ9-TNe5NndF-FQpXVibuvYL5sonTIsUGvuc2bXM6l-GWNv0sxkUKOdk9lwialkIEukL5GRPSTH2zD82yDkkluEDVxCiEad161_mp1BIIQbJp
ozsdMUmr2UO6AoErZX9WkITuvIZRZ1RB6VT2ytYMjj3JUDMLjdPFe_X8pKm1GQsBGCrGJ9zkliqvN5gHSEuNJ3BocKi0QvoYk4zXUnngCv6GdkRTxFB
FYeDxiRlMQyBjquwFwa4i4QMgwL3LnOME91Iezjew1vCy6hRrOUchhkkowy6AEgQTtnFBi4QHlDnpxrNpL3mQcED4L3-JookLrL7okwn99g_eaJzcO5
uQgmz5Gs9W6kgbRTpvB36_vhFkj9T9T-DvrJUA_Kjp111FzSMfEEhjpC_nctXa3DEFgiL1myHgNGwbrx3xKgV-e5xNwEnJhbqitpF4s-3Bi62nKtdlD
TLMgd4VJOomo_fe-U7eEBpdIkoJ9iDkjrODOenAqBkqizmd1OMlLKAOwNYLCNybqEyYU2RdQ2AiBcUAWYiueOwmfqdEuLAAyZUwTpGlAz5Wzo5E3Bey
8r58qmy_3ObpdnGov3h6Q5ZsBUCQeiz_YMidwTFdluDzDE_TARZiMUb8KTNLiX2BTfKHgJRYnDdrKetQC-NkWsOiBGqaVj6S47-D8jIJtFmZEwwnn7W
5SXexAuV8lbm_FBdCVtNJSMkdgQYb4VXgiE9fw7HV2TS6CYKlWQ-S8EFyNLs4MIbTFxFkktd53CckvgKqUsJP_CG_fQt-YvvPI-lAuiqQaECLuGbLgj
5m6XTBuRyUg_c5SriLPnvRNi4hiubl6oJvXIX9eIZFMd7AmlzOtW1KLILpGYX-niiVA-aamgktKKLjdiaIkpu1Oeo_iFWnz-sR3bKqD35VHWPdN1ypj
4lrlqW9wSsUL1hkpl2-B2DqebdMILbLuOJsLsumWxmoi9nNuI7dsMaTZjq7z5CJEknEiH2BoMYj2JPdxaYbuQkjYJjJV46bMBBjcDaUwR.eyJlbmRwb
2ludF9zbHVnIjoib3BlbmFpL2dwdC01LjYtbHVuYS1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_0ea4a64a172e5754016a808893f74c8194a7ad8c0709b1ec96',
                        'index': 1
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786808464-l479n8Xxee6iNTGJKn84',
                'created': 1786808464,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0003059,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0001338,
                    'upstream_inference_prompt_cost': 0.0001721,
                    'upstream_inference_cost': 0.0003059
                }
            },
            id='lc_run--01a00615-6f58-7d03-ae95-05ce713951b3-0',
            tool_calls=[
   

In [10]:
state = {
    "messages": [{"role": "user", "content": "do a private search on politics"}],
    "authenticated": True
}

result = agent.invoke(state)
rprint(result)

{
    'messages': [
        HumanMessage(
            content='do a private search on politics',
            additional_kwargs={},
            response_metadata={},
            id='fcf6a24a-9ad5-480b-994d-7387f30ceaa3'
        ),
        AIMessage(
            content='I can search for political topics, but I can’t guarantee a private or confidential 
search—queries may be processed by the service. What political topic, person, country, or issue would you like me 
to search?',
            additional_kwargs={
                'reasoning_content': '**Clarifying search privacy**\n\nI need to clarify that when it comes to 
"private search," we can\'t guarantee complete privacy. I can offer general search assistance and ask the user 
about their topic instead. Maybe I should use the public search tool only if they specify that\'s what they want. I
just need to explain that we don’t have a private or confidential mode in our search tool, so it\'s all public 
search functionality.',
                'reasoning_details': [
                    {
                        'summary': '**Clarifying search privacy**\n\nI need to clarify that when it comes to 
"private search," we can\'t guarantee complete privacy. I can offer general search assistance and ask the user 
about their topic instead. Maybe I should use the public search tool only if they specify that\'s what they want. I
just need to explain that we don’t have a private or confidential mode in our search tool, so it\'s all public 
search functionality.',
                        'type': 'reasoning.summary',
                        'format': 'openai-responses-v1',
                        'index': 0
                    },
                    {
                        'data': 
'gAAAAABqgIzzTZAes2VzkuIZySLvogWmBrHGxwp4Xsz4w9ScxlPmU0vplUyTAHt1T3M6S_Sf_IjqNVWLrMCHGrBeWrOKsF7a67r4bZcYmJHX1BKgC1
zDgDuIVTn8io1SR4MEdw2sjBtMJtzJNQPTBBjXQNLRGy3SpkrqRkRq4NRmtdNnKXRf1MDXkmum9zJusjWLZonk8wf-YLO5s_3zaIRJWTDrXSy0UnYII
e7OYUujhzQoPR56ccUOen58GHE8FCWg1HQiNkfDznSzlVmVgodjCQ-Y3T7YIRrPK3GNACyFDrw4V4tVa6Gyuh8RE4mtTchwTOb8FgnpXc34T8SAkJqQ
cwZLdUrTmDPuLFmIbpvMUqRQFp17uUPFlSvH5cW3ydW6qq1uECXWQkW__2WO-8pnswloHc84bHxSRO4ayzJpLY968rufsL2E_o2AFnejmzw1qFG07Ku
ZZhp1H83M7hr1DGvcyyoY7WIIUQQVnJi4YVHh8I5BhH-2lFfm4RCfzAAtX9gCVgErdLtM8Vq950Bv_u2icFPCmVjQfB2_xdHzgUVNKPLV5zn8uimLkl
PkQnMlUfpCbq2jO5mjpCfGHXdYZJymhagqF8_Gha5fYi-MPT7FKTxaWnYWDGAY2vy7g5Y8jsQ6GmE_GHZBrAJeLmfWElAvwxmx1fl2PNN-ADhj97SDT
_DcnqCQFKJErkzUqZX3d6No1EUb6fHDcVyI0M2d9qpSBrERXmsD8pU3cwUNKgcMYxC-aNz8bXjyV35WU8PJVbbDYHre2eY_KqhuVRODnQUaz1ItLwZI
fMuFja7gG3aLRZ41NwNQkrKznbSBy0ydM8iWYLYsFgPpeSYaE0T8M6GNkORdKIObj19juEvO_uhArOsziFOv-LvAWiiMXDXUITRtW0HOqQ5Yf_HB6CY
H7trI3XvopH-B7cIZ5vzNLqJCVuiPCG-ztZoMg3uN49CaleyaGIsYiKTLPlrZLaFpejKdqEeq5WDFTvr8fE5aSuouRple44ksk-vb-KxHLP9q-3oGrp
vGPsniz6MzE3ylJus0P_WQNTB1fSDRzTH7NbIDJzzAgwIPRYcN1U-R_SiYSTmoK5NYJn4b6KB44xPs1cQuGxgGnXMRv8XaYj3Wg4JPQ9EQecLpxejuB
ApbevvTMvangjV4tKSU8Pbj8QZb2PEIwV_ixrRzyU_W1bT3E9L784FYLVoqJxo9JtRhNCQdr_PEjGHWo6CVwYazeSs1QMCGWkMSMrN0m5kDdXkklp7w
dczEZUsc9RwBFZofsDcDhExPhm-yeN33i7J6Au7WHsFGhLO_G4YoeFvhU2eHrDbYMNubtCpU_1wnfUQSieS5UKLtQ7kZK45xET3PeerYUytjuW7m5G9
122pRYfOL4Yzcic3mzH06NFPP2o99niSVOKy4XM9Xavjcuokz1h-7F4LhMbfZHBfWQhbcZpCJ2-XxJYBcX9DYoDFhUIgg6v9XC7UONhT-EN6Uryybp6
hxS7fj_nExCqvTWz_ryNtsZZnOF1fTCWUR6W8hK2POcMVG4FBKq_tau0FOEdM0a3lvvJCudbc7H2VpEv39xYKZ4UiSKhMFC4Efchn8YnqObX9-KNFjq
5iC3AUFGWX2ka9I_g-OsNrt38DcADEeB2-h5g2ikv488A0=.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dwdC01LjYtbHVuYS1wcm8tMjAyNjA3MDl
8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_0bb50613fd3d91ab016a808cf290a08196b5e16a6a489e613b',
                        'index': 1
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786809582-iRYImJwnvsiBzoDNy88s',
                'created': 1786809582,
                'ob

In [11]:
# Pass the required initial state with authenticated=True
result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "Search for my private documents"}],
        "authenticated": True,
    }
)

rprint(result)

{
    'messages': [
        HumanMessage(
            content='Search for my private documents',
            additional_kwargs={},
            response_metadata={},
            id='21c414f1-0ded-47d0-9ce7-43096de16f57'
        ),
        AIMessage(
            content='I can help search private documents if you provide access through an available file source or 
upload the files here. I can’t access your device, cloud storage, email, or private accounts on my own.\n\nPlease 
specify:\n- **Where** the documents are (e.g., uploaded files, Google Drive, Dropbox, local folder)\n- **What to 
search for** (keywords, document type, date, author, etc.)',
            additional_kwargs={
                'reasoning_content': "**Clarifying access and location**\n\nI’m realizing I need to clarify what 
access or locations are available for the user. I can only search public sources and won’t be able to look at 
private documents unless the user provides them, or if the connected storage isn't available. So, it’s important to
ask the user where they want me to look and exactly what they're looking for. This will help me assist them 
better!",
                'reasoning_details': [
                    {
                        'summary': "**Clarifying access and location**\n\nI’m realizing I need to clarify what 
access or locations are available for the user. I can only search public sources and won’t be able to look at 
private documents unless the user provides them, or if the connected storage isn't available. So, it’s important to
ask the user where they want me to look and exactly what they're looking for. This will help me assist them 
better!",
                        'type': 'reasoning.summary',
                        'format': 'openai-responses-v1',
                        'index': 0
                    },
                    {
                        'data': 
'gAAAAABqgI19eHhpwDYfdwsngw3vr4XEsfCz6lnOnw2rI-9Ia8-JuwFXWOtc1EF2eNHI-Ydqea8xdEJiBfiI0NxuiqMW2OQbWsBqmD17CjKOkMswF1
WruYQFzndX71fkKTARIqzx2-aU0q-ByO1bjqz0GQH--MPoO_t0rhyOOdMGBUmONrntG19EvT0YN3J7afcrNPLo0TH_0WyY49m3uM0qdqm53hViUrjUb
qBKUY5PvEg_HDsOMQszTsyutpRuMYMSQH0uWCJCnGuuzz4LOp9reuYO3sKpOeewJesDpWDgIxsj0xUvD8Uo-593OA7XN8PVqa71ZWrvTBU2EL1sm_yE
qJAOnWiiT9TD6aufRLtvJdVt54UsaKQcCc1rklNMCj4sTL7Xa2inTMIoG35Tt8qFWwucITm6xoft9no_2dAASQQMJDdw12MKlgAgMc-Eoph4502IAPe
LwX93F25xevs2xSEo6VgAJSG1EV0nAX4GPjxyFLB3CLlxo2ATblR5hyH6BZNaT7IBta10id5lZzFsoQ9_GZTLrnPNOcihHSOjQ9-8ttpBctZ3i6B9J4
SMdZ6Z18IDHLmVhQLel83w5YRhzgae8rzCkitMCsLemY7CDPdppLVrDhFj8rB9RtYpskOgu5GLHRsTiRpK58IRoc6kGq8-GlyB_CSUNbfDBjyiWx997
Lnw2cBQF5c44R4pz3dIxn0JdMDdis0GoDfMDI_qthwN-ehOTBBdvs27hZ02WPgap0f-mrv-aJSOYYsum6wXp-Eu2JrPN0kl5q1y_4eKnwFpKLrRywfp
x_zrrqYFY2HiKoFxZDM_mWlp0jWn2aQECtIanqrd_1zEbQAcTzMFDAqoVKCtdZsFP6LL0oW6XamZnA0W9Vp8mv2ZsWPtX4KW4IJs0aJePI6RbAVIjgm
6vTl1L0YnXSpgK2zhCFnXUZWfmyuBBiBBGZbl8qJQ4MwTmlYHDjypyazmYL9bKbSY3LMFvVr0H_jiYt8D8tl-7_AZek3ga3EehANgxNTRnmDrtZuEiF
BPbQYu8mu9x9CSIJg5V7AM83dHAldusa4w-Uf84nk7GfZtzkpIRt4G8YvDaPbpiXkc1wBeU9Hcv-7okFICkWKEOFa7KJxEVKZvlG34fe7EzvW1yGPHa
VVze_j8nGQXzKJtNVkH5NMbt08g9dDJoli_zAsNV0QqElgowkZUAEwnUStAOV7gkEuMIVNjPlnBAFhNVu8z6eWNyBB9CRcGK-ZA-ZqPA0RL6UpEJ52P
1QEi6-ORiMs6xpXhkxK7gLsfLEzN5l9MFG084Cdn--sjLerudpNaDXk2fRop-Kda-iH6oy72bMRl7XpmxWPZWJNkV8sWZaUEdrNJjj26O9OTS-Khd0X
uy1jLmI_-Re84mmDbQSrfJvSr9r7tJp_CPu9ATL2czTTF5ikxekC85AhHSq-WH4UZoop3q_wVEI1j2QH6YqpoVdjIuiLRTgihGUBx-wc6tHG2-vZ3Hy
-3u8ZRSZgM91RG1N8RkmVCmWUJatkJ-_eJm-9v0xyv8Rgb34QWvmGhyjNrh0DUa-sjbujizXLv6w==.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dw
dC01LjYtbHVuYS1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_0834fd5dffe2f874016a808d7cbbe08195b02b9fbed7690da5',
                        'index': 1
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786809720-qD60e5uSrkOBmowLVV7D',
  